# Generate trips.txt

Generates GTFS trips file linking routes, services, and shapes.

In [1]:
import json
from pathlib import Path 
import pandas as pd
import geopandas as gpd

## Parameters

In [2]:
# Path to params.json (same directory as this notebook)
_params_path = "../params.json"
with open(_params_path, encoding="utf-8") as f:
    p = json.load(f)

In [3]:
CITY = p["city"]
AGENCY_NAME = p["agency"]["name"]
AGENCY_ID = p["agency"]["id"]
AGENCY_URL = p["agency"]["url"]
AGENCY_TIMEZONE = p["agency"]["timezone"]
AGENCY_LANG = p["agency"]["lang"]
START_DATE = p["calendar"]["start_date"]
END_DATE = p["calendar"]["end_date"]
SERVICE_ID = p["calendar"]["SERVICE_ID"]

## Parámetros

In [4]:
# Calendar and service_id parameters loaded from params.json in the previous cell

In [5]:
# Parámetros de calendar y service_id cargados desde params.json en la celda anterior

In [6]:
# --- Carpeta GTFS ---
PATH_DIR_GTFS = Path(f"../data/{CITY}/gtfs-frequencies")
PATH_DIR_GTFS.mkdir(parents=True, exist_ok=True)
print(f"Salida: {PATH_DIR_GTFS.absolute()}")

# --- Carpeta proccesed ---
PATH_DIR_proccesed = Path(f"../data/{CITY}/processed")
PATH_DIR_proccesed.mkdir(parents=True, exist_ok=True)
print(f"Salida: {PATH_DIR_proccesed.absolute()}")

Salida: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/gtfs-frequencies
Salida: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/processed


## Read files

In [7]:
stop_times_df = pd.read_csv(PATH_DIR_GTFS / "stop_times.txt")
stop_times_df.head()

,trip_id,timepoint,stop_id,stop_sequence,arrival_time,departure_time
0,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0000,1,00:00:00,00:00:12
1,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0001,2,00:00:47,00:00:59
2,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0002,3,00:01:34,00:01:46
3,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0003,4,00:02:22,00:02:34
4,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0004,5,00:03:09,00:03:21


In [8]:
routes_gdf = gpd.read_file(PATH_DIR_proccesed / "routes_clean.geojson")
routes_gdf.head(2)

,route_name,route_name_short,route_type,agency_id,shape_id,geometry
0,2_42 Caseta,42 Caseta,3,Red_de_Transporte_Merida,Shape_2_42 Caseta,"LINESTRING (1478265.739 2346761.139, 1478081.3..."
1,6_42 Sur Imss,42 Sur Imss,3,Red_de_Transporte_Merida,Shape_6_42 Sur Imss,"LINESTRING (1478238.108 2346765.115, 1478079.2..."


## Construcción de trips.txt

A partir de los viajes únicos en `stop_times_df` (trip_id, route_name), se asigna un único `service_id`, se incorpora `shape_id` desde `routes_gdf` y se arma la tabla GTFS: route_name, service_id, trip_id, trip_headsign, direction_id, shape_id.

In [9]:
stop_times_df["route_name"] = "Route_" + stop_times_df["stop_id"].str.split("_").str[1]
stop_times_df

,trip_id,timepoint,stop_id,stop_sequence,arrival_time,departure_time,route_name
0,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0000,1,00:00:00,00:00:12,Route_52 Norte Villas La Hacienda R-1
1,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0001,2,00:00:47,00:00:59,Route_52 Norte Villas La Hacienda R-1
2,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0002,3,00:01:34,00:01:46,Route_52 Norte Villas La Hacienda R-1
3,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0003,4,00:02:22,00:02:34,Route_52 Norte Villas La Hacienda R-1
4,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0004,5,00:03:09,00:03:21,Route_52 Norte Villas La Hacienda R-1
...,...,...,...,...,...,...,...
6450,F3_Acim (F.U.T.V.)_trip_00,1,F3_Acim (F.U.T.V.)_0120,121,01:34:46,01:34:58,Route_Acim (F.U.T.V.)
6451,F3_Acim (F.U.T.V.)_trip_00,1,F3_Acim (F.U.T.V.)_0121,122,01:35:33,01:35:45,Route_Acim (F.U.T.V.)
6452,F3_Acim (F.U.T.V.)_trip_00,1,F3_Acim (F.U.T.V.)_0122,123,01:36:21,01:36:33,Route_Acim (F.U.T.V.)
6453,F3_Acim (F.U.T.V.)_trip_00,1,F3_Acim (F.U.T.V.)_0123,124,01:37:08,01:37:20,Route_Acim (F.U.T.V.)


In [10]:
# Viajes únicos: una fila por (trip_id, route_name)
trips_base = (
    stop_times_df[["trip_id", "route_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# route_name debe coincidir con routes.txt (p. ej. Route_11_A, Route_11_B, no Route_11)
# Se deriva del trip_id: "Route_11_A_trip_00" -> "Route_11_A"
trips_base["route_name"] = trips_base["trip_id"].str.replace("_trip_00$", "", regex=True)

# shape_id desde routes_gdf (route_name → shape_id)
routes_shape = routes_gdf[["route_name", "shape_id"]].drop_duplicates()
trips_base = trips_base.merge(routes_shape, on="route_name", how="left")

# Asignar service_id único y columnas opcionales GTFS
trips_base["service_id"] = SERVICE_ID
trips_base["trip_headsign"] = ""
trips_base["direction_id"] = 1

# Orden de columnas para trips.txt
cols_trips = ["route_name", "service_id", "trip_id", "trip_headsign", "direction_id", "shape_id"]
trips_df = trips_base[cols_trips]

trips_df.head(10)

,route_name,service_id,trip_id,trip_headsign,direction_id,shape_id
0,100_52 Norte Villas La Hacienda R-1,standard-schedule,100_52 Norte Villas La Hacienda R-1_trip_00,,1,Shape_100_52 Norte Villas La Hacienda R-1
1,112_Mulsay Juan Pablo Ii,standard-schedule,112_Mulsay Juan Pablo Ii_trip_00,,1,Shape_112_Mulsay Juan Pablo Ii
2,114_Carranza,standard-schedule,114_Carranza_trip_00,,1,Shape_114_Carranza
3,115_San Lucas,standard-schedule,115_San Lucas_trip_00,,1,Shape_115_San Lucas
4,11_50 Penal Paso Texas,standard-schedule,11_50 Penal Paso Texas_trip_00,,1,Shape_11_50 Penal Paso Texas
5,121_Pensiones,standard-schedule,121_Pensiones_trip_00,,1,Shape_121_Pensiones
6,"124_Chichí Suárez, Sitpach",standard-schedule,"124_Chichí Suárez, Sitpach_trip_00",,1,"Shape_124_Chichí Suárez, Sitpach"
7,126_63 Periferico-San Camilo,standard-schedule,126_63 Periferico-San Camilo_trip_00,,1,Shape_126_63 Periferico-San Camilo
8,128_67 Mulsay R-1,standard-schedule,128_67 Mulsay R-1_trip_00,,1,Shape_128_67 Mulsay R-1
9,132_79 Aviación,standard-schedule,132_79 Aviación_trip_00,,1,Shape_132_79 Aviación


## Export

In [11]:
trips_df.to_csv(PATH_DIR_GTFS / "trips.txt", index=False)
print(f"Escrito: {PATH_DIR_GTFS / 'trips.txt'} ({len(trips_df)} viajes)")

Escrito: ../data/merida/gtfs-frequencies/trips.txt (56 viajes)
